In [ ]:
%load_ext autoreload
%autoreload 2

# Mapping TissueTag annotations to traget object

### This notebook illustrates how to migrate TissueTag annotations to visium spot space in the form of an AnnData object but can also be used to match annotations to any type of spatial data in the form of a pandas DF

1) We will load the TissueTag annotated image of the visium dataset and translate the pixel level annotations to an hexagonal binned grid.
2) we will measure the minimal euclidean distances to the annotations and calculate 2 types of biological axes(OrganAxis).
3) Finally we will migrate annotations to visium space and print some plots.

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import figure
import tissue_tag as tt
import tissue_tag.annotation
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

# Part 1 - Load annotation names and colors 

In [ ]:
# set path
path_to_tissue_tag = ".."
path = path_to_tissue_tag + '/data/tissue_tag_minimal_example_visium/' 

In [ ]:
from PIL import Image

# load tissue annotations from TissueTag and migrate annotation to a15um hexagonal grid and outputs the coordinates in ppm=1
tt_object = tt.load_annotation(file_path=path + '/tissue_annotations/annotations.h5')

# Use existing annotation from Nadav
tt_object.label_image = np.array(Image.open(path + '/tissue_annotations/annotations.tif'))

tt_object.grid = tt.generate_grid_from_annotation(tt_object, grid_unit_size= 15)
tt_object.grid.info()

In [ ]:
tt_object.grid['annotation'].value_counts()

In [ ]:
import scanpy as sc
adata = sc.read_visium(path, count_file='raw_feature_bc_matrix.h5')

In [ ]:
tissue_tag.annotation.assign_annotation_label_to_positions(tt_object)
tt_object.positions

In [ ]:
cell_annotation = tt_object.positions["annotation"].to_dict()
adata.obs["annotation"] = adata.obs.apply(lambda r:cell_annotation.get(r.name, None), axis=1)

In [ ]:
adata.obs["annotation"].value_counts(dropna=False)

In [ ]:
# plot the newly annotated visium AnnData
sc.set_figure_params(figsize=[10,10],dpi=75)
sc.pl.spatial(adata,color=['annotation'] ,cmap='gist_rainbow')

In [ ]:
tissue_tag.annotation.plot_labels(tt_object, alpha=0.6)